# Exp11.0 — paired D0/D1 RSNN history internalization

Aggregation-only notebook for `d0_d1_l1mem2_rsnn_fusion_internalization_v2`. D0 and D1 use paired sample geometry, seeds, common initialization streams, and loader order.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    p = Path.cwd() if start is None else Path(start)
    for candidate in (p, *p.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('repo root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_11_0_rsnn_history_internalization' / 'd0_d1_l1mem2_rsnn_fusion_internalization_v2'
runs = pd.read_csv(root / 'run_metrics.csv')
methods = pd.read_csv(root / 'method_summary.csv')
contrasts = pd.read_csv(root / 'paired_contrast_summary.csv')
interactions = pd.read_csv(root / 'interaction_summary.csv')
gaps = pd.read_csv(root / 'temporal_gap_summary.csv')
activity = pd.read_csv(root / 'activity_summary.csv')
d0_sources = pd.read_csv(root / 'd0_source_metrics.csv')
runs


## Native D0/D1 performance


In [ ]:
cols = ['variant','l1_init','topology','seed','native_test_ba','window_test_ba','output_lif_test_ba','fusion_comm_valid_whole_ba','fusion_comm_valid_fixed250_ba','fusion_comm_valid_temporal_gap']
runs[cols].sort_values(['l1_init','topology','seed','variant'])


## Paired D1-D0 effect
Positive delta means D1 is better than D0 under the exact same init/topology/seed condition.


In [ ]:
d1_minus_d0 = contrasts[contrasts['contrast'] == 'd1_minus_d0'].copy()
d1_minus_d0


## Temporal internalization
Track Fixed250-minus-whole from L1 to RSNN to Fusion separately for D0 and D1.


In [ ]:
gap_view = gaps[['variant','l1_init','topology','layer','support','fixed250_ba_mean','whole_ba_mean','temporal_gap_mean']]
gap_view.sort_values(['variant','l1_init','topology','support','layer'])


In [ ]:
plot_data = gaps[gaps['support'] == 'valid'].copy()
for (variant, l1_init, topology), group in plot_data.groupby(['variant','l1_init','topology']):
    ordered = group.set_index('layer').loc[['l1','rsnn','fusion']].reset_index()
    plt.figure()
    plt.plot(ordered['layer'], ordered['temporal_gap_mean'], marker='o')
    plt.axhline(0, linewidth=1)
    plt.ylabel('Fixed250 BA - Whole BA')
    plt.title(f'{variant} / {l1_init} / {topology}')
    plt.show()


## Recurrence and pretraining interactions


In [ ]:
interactions


## D0 matched-source sanity check


In [ ]:
d0_sources


## Recurrent activity diagnostics


In [ ]:
runs[['variant','l1_init','topology','seed','mean_abs_external_input','mean_abs_recurrent_input','recurrent_to_external_abs_ratio','recurrent_weight_norm']].sort_values(['variant','l1_init','topology','seed'])
